In [22]:
import math
import random
import re

In [23]:
# =========================================================
# 1. BASIC CONFIGURATION
# =========================================================

EMBEDDING_SIZE = 8
NUM_HEADS = 2
HEAD_SIZE = EMBEDDING_SIZE // NUM_HEADS

In [24]:
# =========================================================
# 2. VOCABULARY
# =========================================================

vocab = {
    "[PAD]": 0,
    "[CLS]": 1,
    "[SEP]": 2,
    "i": 3,
    "love": 4,
    "like": 5,
    "this": 6,
    "movie": 7,
    "is": 8,
    "was": 9,
    "very": 10,
    "good": 11,
    "great": 12,
    "amazing": 13,
    "excellent": 14,
    "bad": 15,
    "hate": 16,
    "terrible": 17,
    "boring": 18,
    "worst": 19,
    "awful": 20,
    "not": 21
}

In [25]:
# =========================================================
# 3. TOKENIZATION
# =========================================================

def tokenize(text):

    text = text.lower()

    return re.findall(r"\w+|[.!?,]", text)


def encode(text):

    tokens = tokenize(text)

    tokens = ["[CLS]"] + tokens + ["[SEP]"]

    ids = []

    for token in tokens:

        if token in vocab:
            ids.append(vocab[token])

        else:
            ids.append(0)

    return tokens, ids

In [26]:
# =========================================================
# 4. MATRIX FUNCTIONS
# =========================================================

def matrix(rows, cols):

    return [
        [random.uniform(-0.5, 0.5)
         for _ in range(cols)]
        for _ in range(rows)
    ]


def vector_matrix_multiply(vector, weights):

    result = []

    for j in range(len(weights[0])):

        value = 0

        for i in range(len(vector)):

            value += vector[i] * weights[i][j]

        result.append(value)

    return result
def add_vectors(a, b):

    return [
        a[i] + b[i]
        for i in range(len(a))
    ]


In [27]:
# =========================================================
# 5. TOKEN EMBEDDINGS
# =========================================================

random.seed(42)

token_embeddings = matrix(
    len(vocab),
    EMBEDDING_SIZE
)

In [28]:
# =========================================================
# 6. POSITIONAL ENCODING
# =========================================================

def positional_encoding(position):

    result = []

    for i in range(EMBEDDING_SIZE):

        angle = position / (
            10000 ** (
                (2 * (i // 2)) /
                EMBEDDING_SIZE
            )
        )

        if i % 2 == 0:
            result.append(math.sin(angle))

        else:
            result.append(math.cos(angle))

    return result


In [29]:
# =========================================================
# 7. CREATE INPUT EMBEDDINGS
# =========================================================

def create_embeddings(token_ids):

    result = []

    for position, token_id in enumerate(token_ids):

        token_vector = token_embeddings[token_id]

        position_vector = positional_encoding(
            position
        )

        combined = add_vectors(
            token_vector,
            position_vector
        )

        result.append(combined)

    return result


In [30]:
# =========================================================
# 8. MATRIX MULTIPLICATION FOR Q K V
# =========================================================

WQ = matrix(EMBEDDING_SIZE, EMBEDDING_SIZE)
WK = matrix(EMBEDDING_SIZE, EMBEDDING_SIZE)
WV = matrix(EMBEDDING_SIZE, EMBEDDING_SIZE)


def create_qkv(x):

    Q = vector_matrix_multiply(x, WQ)
    K = vector_matrix_multiply(x, WK)
    V = vector_matrix_multiply(x, WV)

    return Q, K, V


In [31]:
# =========================================================
# 9. SOFTMAX
# =========================================================

def softmax(values):

    maximum = max(values)

    exp_values = [
        math.exp(x - maximum)
        for x in values
    ]

    total = sum(exp_values)

    return [
        x / total
        for x in exp_values
    ]

In [32]:
# =========================================================
# 10. DOT PRODUCT
# =========================================================

def dot_product(a, b):

    total = 0

    for i in range(len(a)):

        total += a[i] * b[i]

    return total

In [33]:
# =========================================================
# Q, K, V MATRIX MULTIPLICATION
# =========================================================

WQ = matrix(EMBEDDING_SIZE, EMBEDDING_SIZE)
WK = matrix(EMBEDDING_SIZE, EMBEDDING_SIZE)
WV = matrix(EMBEDDING_SIZE, EMBEDDING_SIZE)


def create_qkv(x):

    Q = []
    K = []
    V = []

    # x contains multiple token vectors
    for vector in x:

        Q.append(
            vector_matrix_multiply(vector, WQ)
        )

        K.append(
            vector_matrix_multiply(vector, WK)
        )

        V.append(
            vector_matrix_multiply(vector, WV)
        )

    return Q, K, V


# =========================================================
# SOFTMAX
# =========================================================

def softmax(values):

    maximum = max(values)

    exp_values = [
        math.exp(value - maximum)
        for value in values
    ]

    total = sum(exp_values)

    return [
        value / total
        for value in exp_values
    ]


# =========================================================
# DOT PRODUCT
# =========================================================

def dot_product(a, b):

    total = 0

    for i in range(len(a)):
        total += a[i] * b[i]

    return total


# =========================================================
# SELF ATTENTION
# =========================================================

def self_attention(x):

    Q, K, V = create_qkv(x)

    output = []

    scale = math.sqrt(EMBEDDING_SIZE)

    # Process every query
    for q in Q:

        scores = []

        # Compare query with every key
        for k in K:

            score = dot_product(q, k) / scale

            scores.append(score)

        # Convert scores to attention probabilities
        attention_weights = softmax(scores)

        result = [
            0.0
            for _ in range(EMBEDDING_SIZE)
        ]

        # Weighted sum of values
        for weight, v in zip(
            attention_weights,
            V
        ):

            for i in range(EMBEDDING_SIZE):

                result[i] += weight * v[i]

        output.append(result)

    return output

In [34]:
# =========================================================
# MULTI-HEAD ATTENTION
# =========================================================

def multi_head_attention(x):

    heads = []

    for head in range(NUM_HEADS):

        head_output = self_attention(x)

        heads.append(head_output)

    combined = []

    for position in range(len(x)):

        vector = []

        for head in heads:

            vector.extend(
                head[position][:HEAD_SIZE]
            )

        combined.append(vector)

    return combined

In [35]:
# =========================================================
# 13. RELU
# =========================================================

def relu(x):

    return [
        max(0, value)
        for value in x
    ]

In [36]:
# =========================================================
# 14. FEED FORWARD NETWORK
# =========================================================

W1 = matrix(
    EMBEDDING_SIZE,
    EMBEDDING_SIZE * 2
)

W2 = matrix(
    EMBEDDING_SIZE * 2,
    EMBEDDING_SIZE
)


def feed_forward(x):

    hidden = vector_matrix_multiply(
        x,
        W1
    )

    hidden = relu(hidden)

    output = vector_matrix_multiply(
        hidden,
        W2
    )

    return output

In [37]:
# =========================================================
# 15. LAYER NORMALIZATION
# =========================================================

def layer_normalization(x):

    mean = sum(x) / len(x)

    variance = sum(
        (value - mean) ** 2
        for value in x
    ) / len(x)

    return [
        (value - mean) /
        math.sqrt(variance + 1e-6)

        for value in x
    ]

In [38]:
# =========================================================
# 16. TRANSFORMER ENCODER
# =========================================================

def transformer_encoder(x):

    # Self attention
    attention = multi_head_attention(x)

    # Residual connection
    residual1 = []

    for original, attended in zip(
        x,
        attention
    ):

        residual1.append(
            layer_normalization(
                add_vectors(
                    original,
                    attended
                )
            )
        )

    # Feed Forward
    ff_output = []

    for vector in residual1:

        ff_output.append(
            feed_forward(vector)
        )

    # Second residual connection
    final_output = []

    for vector, ff in zip(
        residual1,
        ff_output
    ):

        final_output.append(
            layer_normalization(
                add_vectors(
                    vector,
                    ff
                )
            )
        )

    return final_output

In [39]:
# =========================================================
# 17. SENTIMENT CLASSIFIER
# =========================================================

positive_words = {
    "love",
    "like",
    "good",
    "great",
    "amazing",
    "excellent"
}

negative_words = {
    "hate",
    "bad",
    "terrible",
    "boring",
    "worst",
    "awful"
}


def classify_sentiment(tokens):

    positive = 0
    negative = 0

    for token in tokens:

        if token in positive_words:
            positive += 1

        if token in negative_words:
            negative += 1

    if positive > negative:

        return "POSITIVE"

    elif negative > positive:

        return "NEGATIVE"

    else:

        return "NEUTRAL"

In [40]:
# =========================================================
# 18. MAIN
# =========================================================

print("====================================")
print(" BERT FROM SCRATCH - SENTIMENT")
print("====================================")

text = input("\nEnter sentence: ")

tokens, token_ids = encode(text)

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)


# Embedding
x = create_embeddings(token_ids)

# Transformer Encoder
encoded = transformer_encoder(x)

# [CLS] representation
cls_vector = encoded[0]

print("\n[CLS] Representation:")
print([
    round(value, 4)
    for value in cls_vector
])


# Sentiment
sentiment = classify_sentiment(tokens)

print("\nSentiment:")
print(sentiment)

 BERT FROM SCRATCH - SENTIMENT

Enter sentence: I love this movie

Tokens:
['[CLS]', 'i', 'love', 'this', 'movie', '[SEP]']

Token IDs:
[1, 3, 4, 6, 7, 2]

[CLS] Representation:
[-1.4736, 0.3792, 0.7552, 0.7788, -1.9316, 0.3536, 0.6118, 0.5266]

Sentiment:
POSITIVE
